In [ ]:
# === SETUP ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from scipy.optimize import minimize
from scipy.stats import rankdata
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.special import expit
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pytabkit import TabM_D_Classifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import TargetEncoder
from tqdm import tqdm
from realmlp import RealMLP_TD_Classifier, seed_everything
from utils import*
import warnings
import copy
import time
import gc
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.set_per_process_memory_fraction(0.70, device=0)
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11, 'figure.dpi': 100})

print('Setup complete!')

W0921 13:43:54.511000 9520 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Setup complete!


In [3]:
# === LOAD DATA ===
train_raw = pd.read_csv('dataset/train.csv')
test_raw = pd.read_csv('dataset/test.csv')
original_raw = pd.read_csv('dataset/EV_Adoption_and_Range_Anxiety_Dataset.csv')

print(f'Train: {train_raw.shape}')
print(f'Test:  {test_raw.shape}')
print(f'Original: {original_raw.shape}')

# === FIX THE ORIGINAL DATASET NaN ISSUE ===
# The original has 534 rows with NaN values (out of 10,000).
# Solution: Simply drop NaN rows. We keep 9,466 clean rows.
# This is better than imputing because imputed values could introduce noise.
print(f'\nOriginal NaN rows: {original_raw.isnull().any(axis=1).sum()}')
original_clean = original_raw.dropna().reset_index(drop=True)
print(f'After dropping NaN: {original_clean.shape[0]} clean rows')

# Rename to match train columns
original_clean.rename(columns={'Buyer_ID': 'id'}, inplace=True)

Train: (668665, 15)
Test:  (286571, 14)
Original: (10000, 15)

Original NaN rows: 534
After dropping NaN: 9466 clean rows


In [4]:
def engineer_features_v3(df):
    """Complete feature engineering with error-analysis improvements."""
    out = df.copy()

    # --- ENCODE CATEGORICALS ---
    out['Subsidy_Available'] = (df['Subsidy_Available'] == 'Yes').astype(int)
    out['Home_Charging_Possible'] = (df['Home_Charging_Possible'] == 'Yes').astype(int)
    out['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    out['Gender_M'] = (df['Gender'] == 'Male').astype(int)
    out['City_Urban'] = (df['City_Type'] == 'Urban').astype(int)
    out['City_Suburban'] = (df['City_Type'] == 'Suburban').astype(int)

    # --- TIER 1: Core interactions (from EDA) ---
    out['Subsidy_x_EnvConcern'] = out['Subsidy_Available'] * df['Environmental_Concern_Level']
    out['EnvConcern_sq'] = df['Environmental_Concern_Level'] ** 2
    out['Income_x_Concern'] = np.log1p(df['Annual_Income_USD']) * df['Environmental_Concern_Level']

    # --- TIER 2: Medium impact ---
    out['Total_Charging'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    out['Charging_Ratio'] = df['Charging_Stations_Near_Home'] / (out['Total_Charging'] + 1)
    out['Ready_Score'] = out['Subsidy_Available'] * df['Environmental_Concern_Level'] * (1 - out['Range_Anxiety_Level'] / 2)

    # --- TIER 3: Supporting ---
    out['log_income'] = np.log1p(df['Annual_Income_USD'])
    out['Subsidy_x_HomeCharging'] = out['Subsidy_Available'] * out['Home_Charging_Possible']
    out['Income_per_Car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 0.01)
    out['Commute_x_Anxiety'] = df['Daily_Commute_km'] * out['Range_Anxiety_Level']
    out['Subsidy_x_Income'] = out['Subsidy_Available'] * df['Annual_Income_USD']
    out['Concern_x_NoAnxiety'] = df['Environmental_Concern_Level'] * (2 - out['Range_Anxiety_Level'])
    out['High_Concern_High_Income'] = (
        (df['Environmental_Concern_Level'] >= 4) &
        (df['Annual_Income_USD'] > df['Annual_Income_USD'].median())
    ).astype(int)

    # --- 3-WAY INTERACTIONS ---
    # Error analysis showed the model struggles when 2 features are ambiguous.
    # 3-way products give it more resolution in these zones.
    out['Sub_Concern_Charging'] = (out['Subsidy_Available'] * df['Environmental_Concern_Level']
                                    * out['Home_Charging_Possible'])
    out['Sub_Concern_NoAnxiety'] = (out['Subsidy_Available'] * df['Environmental_Concern_Level']
                                     * (2 - out['Range_Anxiety_Level']))
    out['Income_Concern_Subsidy'] = (np.log1p(df['Annual_Income_USD'])
                                      * df['Environmental_Concern_Level']
                                      * out['Subsidy_Available'])

    # --- POLYNOMIAL FEATURES ---
    out['Concern_cubed'] = df['Environmental_Concern_Level'] ** 3
    out['Ready_Score_sq'] = out['Ready_Score'] ** 2

    # --- RATIO FEATURES ---
    out['Commute_per_income'] = df['Daily_Commute_km'] / (np.log1p(df['Annual_Income_USD']) + 1)
    out['Charging_per_commute'] = out['Total_Charging'] / (df['Daily_Commute_km'] + 1)
    out['Income_x_Sub_x_Concern_ratio'] = (
        (np.log1p(df['Annual_Income_USD']) * out['Subsidy_Available'] * df['Environmental_Concern_Level'])
        / (out['Range_Anxiety_Level'] + 1)
    )

    # --- COMPOSITE "READINESS" SCORES ---
    out['Readiness_v2'] = (
        df['Environmental_Concern_Level'] * 2 +
        out['Subsidy_Available'] * 3 +
        out['Home_Charging_Possible'] * 1 -
        out['Range_Anxiety_Level'] * 2
    )

    out['Readiness_income_weighted'] = out['Readiness_v2'] * np.log1p(df['Annual_Income_USD'])

    # --- DIFFERENCE FEATURES ---
    out['Charging_diff'] = (df['Charging_Stations_Near_Home'] - df['Charging_Stations_Near_Work'])
    out['Concern_minus_Anxiety'] = df['Environmental_Concern_Level'] - out['Range_Anxiety_Level'] * 2.5

    # --- BIN FEATURES ---
    # out['Income_bin'] = pd.qcut(df['Annual_Income_USD'], 20, labels=False, duplicates='drop')
    out['Income_bin'] = pd.cut(
        df['Annual_Income_USD'],
        bins=[0, 30000, 60000, 100000, 200000, float('inf')],
        labels=[0, 1, 2, 3, 4]
    ).astype(float)
    out['Age_bin'] = pd.cut(df['Age'], bins=[0, 25, 35, 45, 55, 65, 100], labels=False)


    out['Is_Charging_0_2'] = out['Charging_Stations_Near_Home'].apply(lambda x: 1 if x<=2 else 0)
    out['Is_Charging_11_14'] = out['Charging_Stations_Near_Home'].apply(lambda x: 1 if x>=11 else 0)

    out['Charging_Combo_0_2'] = (out['Is_Charging_0_2'] * 2 + out['Home_Charging_Possible']).astype('category')
    out['Charging_Combo_11_14'] = (out['Is_Charging_11_14'] * 2 + out['Home_Charging_Possible']).astype('category')

    out['Is_30k_spike'] = (out['Annual_Income_USD'] == 30000.0).astype('int8').astype('category')
    out['Is_millionaire_cliff'] = (out['Annual_Income_USD'] >= 170537.0).astype('int8').astype('category')
    out['is_dead_zone'] = ((out['Annual_Income_USD'] >= 38000.0) & (out['Annual_Income_USD'] <= 42000.0)).astype('int8').astype('category')
    out['Income_100_floor'] = np.floor(out['Annual_Income_USD'] / 100.0).astype('category').astype(int)
    out['Income_1000_floor'] = np.floor(out['Annual_Income_USD'] / 1000.0).astype('category')



    out = pd.get_dummies(out, columns=['Current_Car_Type'], drop_first=True, dtype='int8')
    return out

# Apply to all datasets
train = engineer_features_v3(train_raw)
test = engineer_features_v3(test_raw)
original = engineer_features_v3(original_clean)

# Set targets
train['target'] = (train['Will_Buy_EV'] == 'Yes').astype(int)
original['target'] = (original['Will_Buy_EV'] == 'Yes').astype(int)

# Define feature columns
# drop_cols = ['id', 'Will_Buy_EV', 'target', 'Gender', 'City_Type', 'Current_Car_Type',
#              'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
drop_cols = ['id','Will_Buy_EV','target','Gender','City_Type']
feature_cols = [c for c in train.columns if c not in drop_cols]

# y = train['target'].values
# X = train[feature_cols].values
# X_test = test[feature_cols].values
# X_orig = original[feature_cols].values
# y_orig = original['target'].values

print(f'Features v3: {len(feature_cols)} features')
print(f'New features: {len(feature_cols) - 21} more than original')
print(f'X_orig: {original.shape} (NaN-free original data)')
print(f'\nAll features:')
for i, f in enumerate(feature_cols):
    print(f'  {i+1:2d}. {f}')

Features v3: 52 features
New features: 31 more than original
X_orig: (9466, 57) (NaN-free original data)

All features:
   1. Age
   2. Annual_Income_USD
   3. Daily_Commute_km
   4. Number_of_Cars_Owned
   5. Charging_Stations_Near_Home
   6. Charging_Stations_Near_Work
   7. Environmental_Concern_Level
   8. Home_Charging_Possible
   9. Subsidy_Available
  10. Range_Anxiety_Level
  11. Gender_M
  12. City_Urban
  13. City_Suburban
  14. Subsidy_x_EnvConcern
  15. EnvConcern_sq
  16. Income_x_Concern
  17. Total_Charging
  18. Charging_Ratio
  19. Ready_Score
  20. log_income
  21. Subsidy_x_HomeCharging
  22. Income_per_Car
  23. Commute_x_Anxiety
  24. Subsidy_x_Income
  25. Concern_x_NoAnxiety
  26. High_Concern_High_Income
  27. Sub_Concern_Charging
  28. Sub_Concern_NoAnxiety
  29. Income_Concern_Subsidy
  30. Concern_cubed
  31. Ready_Score_sq
  32. Commute_per_income
  33. Charging_per_commute
  34. Income_x_Sub_x_Concern_ratio
  35. Readiness_v2
  36. Readiness_income_weighted

In [5]:
def add_digit_features(df):
    out = df.copy()
    num_cols = [
        'Age', 'Annual_Income_USD', 'Daily_Commute_km',
        'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
    ]
    for col in num_cols:
        scaled = np.rint(out[col] * 10**4).astype("int64")
        for k in range(-4, 4):
            out[f"{col}_digit{k}"] = scaled // 10**(k + 4) % 10
    return out

print('add_digit_features defined')

def add_freq_features(train_df, test_df, cols, orig_df=None):
    if orig_df is None:
        tr, te = train_df.copy(), test_df.copy()
        for c in cols:
            freq = pd.concat(
                [tr[c].astype(str), te[c].astype(str)], ignore_index=True
            ).value_counts(normalize=True)
            tr[f'{c}_freq'] = tr[c].astype(str).map(freq).astype('float32')
            te[f'{c}_freq'] = te[c].astype(str).map(freq).astype('float32')
        return tr, te
    tr, te, ori = train_df.copy(), test_df.copy(), orig_df.copy()
    for c in cols:
        freq = pd.concat(
            [tr[c].astype(str), te[c].astype(str), ori[c].astype(str)], ignore_index=True
        ).value_counts(normalize=True)
        tr[f'{c}_freq'] = tr[c].astype(str).map(freq).astype('float32')
        te[f'{c}_freq'] = te[c].astype(str).map(freq).astype('float32')
        ori[f'{c}_freq'] = ori[c].astype(str).map(freq).astype('float32')
    return tr, te, ori

print('add_freq_features defined')

add_digit_features defined
add_freq_features defined


In [6]:
num_cols = [
        'Age', 'Annual_Income_USD', 'Daily_Commute_km',
        'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
        'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
    ]

train, test = add_digit_features(train), add_digit_features(test)
original = add_digit_features(original)

train, test, original = add_freq_features(train, test, test.columns.tolist(), orig_df=original)
train, test = train.align(test, join='left', axis=1, fill_value=0)

print(f'Before feature selection: {train.shape[1]} features')

const_cols = [c for c in train.columns if train[c].nunique() == 1]
train = train.drop(columns=const_cols)
test  = test.drop(columns=const_cols)
original  = original.drop(columns=const_cols)

corr_matrix = train[num_cols].corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop     = [col for col in upper_tri.columns if any(upper_tri[col] == 1.0)]
train = train.drop(columns=to_drop)
test  = test.drop(columns=to_drop)
original  = original.drop(columns=to_drop)

print(f'Dropped {len(const_cols)} constant + {len(to_drop)} correlated features')
print(f'Constant Features: {const_cols}')
print(f'Correlated Features: {to_drop}')
print(f'Final shape: {train.shape}')

Before feature selection: 224 features
Dropped 83 constant + 0 correlated features
Constant Features: ['Age_digit-4', 'Age_digit-3', 'Age_digit-2', 'Age_digit-1', 'Age_digit2', 'Age_digit3', 'Annual_Income_USD_digit-4', 'Annual_Income_USD_digit-3', 'Annual_Income_USD_digit-2', 'Annual_Income_USD_digit-1', 'Daily_Commute_km_digit-4', 'Daily_Commute_km_digit-3', 'Daily_Commute_km_digit-2', 'Daily_Commute_km_digit2', 'Daily_Commute_km_digit3', 'Number_of_Cars_Owned_digit-4', 'Number_of_Cars_Owned_digit-3', 'Number_of_Cars_Owned_digit-2', 'Number_of_Cars_Owned_digit-1', 'Number_of_Cars_Owned_digit1', 'Number_of_Cars_Owned_digit2', 'Number_of_Cars_Owned_digit3', 'Charging_Stations_Near_Home_digit-4', 'Charging_Stations_Near_Home_digit-3', 'Charging_Stations_Near_Home_digit-2', 'Charging_Stations_Near_Home_digit-1', 'Charging_Stations_Near_Home_digit2', 'Charging_Stations_Near_Home_digit3', 'Charging_Stations_Near_Work_digit-4', 'Charging_Stations_Near_Work_digit-3', 'Charging_Stations_Near_

In [ ]:
# === FINAL, STABLE TRAINING STACK ===

# Remove broken / redundant columns before model fitting.
FEATURES = [c for c in train.columns if c not in drop_cols]
constant_cols = [c for c in FEATURES if train[c].nunique() <= 1]
FEATURES = [c for c in FEATURES if c not in constant_cols]

corr = train[FEATURES].sample(min(30000, len(train)), random_state=42).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant_cols = [c for c in upper.columns if any(upper[c] > 0.96)]
FEATURES = [c for c in FEATURES if c not in redundant_cols]
cat_cols = [c for c in FEATURES if str(train[c].dtype) == 'category']
TE_COLS = [c for c in FEATURES if cat_cols]
TE_COLS += ['Income_100_floor', 'Income_1000_floor']
FEATURES += ['Income_100_floor', 'Income_1000_floor']

print(f'Features kept: {len(FEATURES)}')
print(f'Redundant features removed: {len(redundant_cols)}')


lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.005,
    'num_leaves': 68,
    'max_depth': 9,
    # 'num_leaves': 63,
    # 'max_depth': 7,
    'min_child_samples': 42,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 2.0,
    'scale_pos_weight': 4.7,
    'verbosity': -1,
    'n_estimators': 30000,
    'device': 'gpu' if torch.cuda.is_available() else 'cpu',
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'eta': 0.02,
    'max_depth': 6,
    'min_child_weight': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.5,
    'alpha': 0.1,
    'lambda': 1.5,
    'scale_pos_weight': 4.7,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    # 'tree_method': 'gpu_hist' if torch.cuda.is_available() else 'hist',
    'tree_method': 'hist',
}

cat_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'learning_rate': 0.03,
    'depth': 9,
    # 'depth': 7,
    'l2_leaf_reg': 3.0,
    'bagging_temperature': 1.0,
    'random_strength': 1.0,
    'min_data_in_leaf': 30,
    'border_count': 128,
    'od_type': 'Iter',
    'od_wait': 300,
    'iterations': 8000,
    'task_type': 'GPU' if torch.cuda.is_available() else 'CPU',
}
tabm_params = {
    "verbosity": 0,
    # "arch_type": "tabm-mini-normal",
    "arch_type": "tabm-normal",
    "num_emb_type": "pwl",
    "tabm_k": 24,
    "d_embedding": 32,
    "dropout": 0.12,
    "batch_size": 512,
    "lr": 1e-3,
    "n_epochs": 100,
    "d_block": 256,
    "n_blocks": 5,
    "patience": 2,
    "weight_decay": 3e-3,
    "device": "cuda",
}
CLASSES = [0,1]
seed_everything(42)

realmlp_params = {
    'n_ens': 8,
    'embed_dim': 7,
    'onehot_thresh': 10,
    'hidden_dims': [512, 512, 512],
    'dropout': 0.044,
    'p_drop_sched': 'expm4t',
    'activation': nn.GELU,
    'add_front_scale': True,
    'pbld_hidden_dim': 16,
    'pbld_out_dim': 5,
    'pbld_freq_scale': 2.33,
    'pbld_activation': nn.PReLU,
    'pbld_lr_factor': 0.115,
    'lr': 0.01,
    'mom': 0.9,
    'sq_mom': 0.98,
    'lr_sched': 'flat_cos',
    'flat_ratio': 0.20,
    'first_layer_lr_factor': 1.0,
    'first_layer_wd_factor': 0.1,
    'lr_scale_mult': 10.0,
    'lr_bias_mult': 0.1,
    'weight_decay': 0.0125,
    'wd_scale_mult': 0.1,
    'wd_bias_mult': 0.5,
    'grad_clip': 1.0,
    'class_weight_power': 0.0,
    'class_weight_multipliers': None,
    'sample_weight_power': 0.0,
    'loss_prior_power': 1.075,
    'focal_gamma': 0.0,
    'eval_class_multipliers': None,
    'ls_eps': 0.04,
    'ls_eps_sched': 'cos',
    'tfms': ['median_center', 'robust_scale'],
    'epochs': 7,
    'train_bs': 256,
    'eval_bs': 10240,
    'numeric_noise_std': 0.0,
    'ema_decay': 0.997875,
    'verbosity': 2,
    'use_early_stopping': False,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,
    'device': str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')),
    'random_state': 42,
}
lr_params_default = {
    'penalty': 'l2',
    'C': 0.1,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'n_jobs': -1,
    'class_weight': 'balanced',
}
lr_params_elasticnet = {
    'penalty': 'elasticnet',
    'C': 0.05,
    'l1_ratio': 0.5,
    'solver': 'saga',
    'max_iter': 2000,
    'tol': 1e-3,
    'n_jobs': -1,
    'class_weight': 'balanced',
}


# Use a small, robust seed set for variance reduction.
seeds = [42, 101, 2026]

y = train['target']

Features kept: 80
Redundant features removed: 56


In [ ]:
lgb_oof, lgb_test = seed_average(train_cv_lgbm, train, y, test, FEATURES, lgb_params, seeds, TE_COLS=TE_COLS)

# Seed   42 | AUC=0.945659
# Seed  101 | AUC=0.945648
# Seed 2026 | AUC=0.945676
# Average OOF AUC: 0.945903

In [ ]:
xgb_oof, xgb_test = seed_average(train_cv_xgb, train, y, test, FEATURES, xgb_params, seeds, TE_COLS=TE_COLS)

# Seed   42 | AUC=0.945765
# Seed  101 | AUC=0.945704
# Seed 2026 | AUC=0.945725
# Average OOF AUC: 0.945987

In [ ]:
cat_oof, cat_test = seed_average(train_cv_catboost, train, y, test, FEATURES, cat_params, seeds, TE_COLS=TE_COLS)

# Seed   42 | AUC=0.945490
# Seed  101 | AUC=0.945465
# Seed 2026 | AUC=0.945484
# Average OOF AUC: 0.945725

In [ ]:
tabm_oof, tabm_test = seed_average(train_cv_tabm, train, y, test, FEATURES, tabm_params, seeds, TE_COLS=TE_COLS)

# Seed   42 | AUC=0.945730
# Seed  101 | AUC=0.945719
# Seed 2026 | AUC=0.945619
# Average OOF AUC: 0.945974

In [ ]:
oof_dict = {'lgbm': lgb_oof, 'xgb': xgb_oof, 'catboost': cat_oof, 'tabm':tabm_oof}
test_dict = {'lgbm': lgb_test, 'xgb': xgb_test, 'catboost': cat_test, 'tabm':tabm_test}

final_oof, final_test, final_weights = rank_blend(oof_dict, test_dict, y)

submission = pd.DataFrame({
    'id': test['id'],
    'Will_Buy_EV': final_test
})
submission.to_csv('submission_blend_rank_opt.csv', index=False)
print('Saved ranked ensemble submission: submission_blend_rank_opt.csv')
print(f'Final CV AUC: {roc_auc_score(y, final_oof):.6f}')

oof_save = pd.DataFrame(oof_dict)
oof_save['target'] = y
oof_save.to_csv('oof_predictions_v5.csv', index=False)

test_oof_save = pd.DataFrame(test_dict)
test_oof_save.to_csv('test_oof_predictions_v5.csv', index=False)

# Optimal rank-blend weights:
#   lgbm: 0.0008
#   xgb: 0.4996
#   catboost: 0.0260
#   tabm: 0.4735
# Blend OOF AUC: 0.946075
# Saved ranked ensemble submission: submission_blend_rank_opt.csv
# Final CV AUC: 0.946075